# Discrepancy explorer

Interactive view of FSS collapse under discrepancy models (noise, additive $a\cdot g$, z-threshold, FSS) and GP couplings (additive / mixture / weighted).

Likelihood channels are the **m / m² / m⁴ / U₄** checkboxes (recompiles). Default is $m+m^2$. $\beta$ is identifiable from magnetization moments, not from Binder alone. The collapse plot is still $\Phi_m$ (point selection); heatmaps use the selected LML channels.

**Noise inflation** and **additive $a\cdot g$ GP** share
$$
a(t,L)=\Bigl(\frac{|t|}{t_0}\Bigr)^p+\Bigl(\frac{L_0}{L}\Bigr)^q.
$$

**Noise inflation** (legacy):
$$\sigma_{\mathrm{eff}}=\sqrt{(s_\sigma\,\sigma_{\mathrm{MC}})^2+(\sigma_{\mathrm{model}}\,K\,a)^2}.$$
**$s_\sigma$** (slider) multiplies every MC error bar uniformly — a test of “are the reported $\sigma_{\mathrm{MC}}$ too small?” Distinct from discrepancy $a$, which is gated in $(t,L)$. MCMC: `--obs-sigma-scale`.

**Additive GP** (slide form; $\Phi_1$ dropped, $r=g$):
$$m=L^{-\beta/\nu}f(z)+a\,g(z)+\varepsilon_m
\quad\Rightarrow\quad
\Phi_m=f+(a\,L^{\beta/\nu})g+\varepsilon_\Phi,$$
and $\Phi_{m^2}=f+(a\,L^{2\beta/\nu})g+\varepsilon$ (shared raw $a$).  Independent GPs
$f\sim\mathrm{GP}(0,\eta^2 k_{\ell_f})$ (universal) and $g\sim\mathrm{GP}(0,\sigma_g^2 k_{\ell_g})$.
Sliders **ℓ_f** / **η_f** set the main GP on $f$; **ℓ_g** is the length scale of $g$ (also in $z$-units; previously locked to $\ell_f$); **σ_g** (same control as σ_model) sets the discrepancy amplitude — try large values to absorb gated points.

**Profile ℓ_f** (recommended): each heatmap cell is $\max_{\ell_f}\log L$, so the exponent contours do not depend on a hand-chosen length scale. The bottom-right panel is $\Delta\log L(\ell_f)$ at exact $(T_c,\nu,\beta)$ — a sharp peak means the data identify $\ell_f$; a flat curve means they do not. MCMC should infer $\ell_f$ (`--infer-gp-hyperparams`) rather than pin it.

**κ likelihood** (FSS form): $\Delta\log L(\kappa)$ at exact $(T_c,\nu,\beta)$ is always shown, with $\omega$ and $\ell_f$ pinned to the sliders (or max over $\ell_f$ if that box is on). The green diamond is $\kappa^*$; a sharp peak means $\kappa$ is identified. Heatmaps still use the slider $\kappa$.

**Z-threshold additive GP**: collapsed amplitude
$a=1_{|z|>10}$ (else $0$), no $L^{\beta/\nu}$ factor.  Large $\sigma_g$ should nearly remove those points' pull on $f$ and $\nu$.

**FSS additive GP**: collapsed amplitude (no $L^{\beta/\nu}$)
$$
a=L^{-\omega}+\kappa\,|t|^{\omega\nu}=L^{-\omega}\bigl(1+\kappa\,|z|^{\omega\nu}\bigr),
$$
so $\Phi=f+a\,g+\varepsilon$.  Sliders **ω** and **κ** ($κ=0$ is pure $L^{-\omega}$).  Heatmaps recompute $a$ at each trial $(T_c,\nu)$.

**Coupling** (GP forms; ignored for noise inflation). The amplitude $a$ may be $>1$; the gate is $\pi=a/(1+a)$ (z-threshold already has $a\in\{0,1\}$, so $\pi=a$).

- **additive** (default): $\Phi=f+ag+\varepsilon$ — every point still trains $f$.
- **mixture**: $\Phi=(1-\pi)f+\pi g+\varepsilon$ — out-of-window points train $g$, not $f$.
- **weighted**: $\Phi=(1-\pi)f+\varepsilon$ — no second GP; $K_{ij}=(1-\pi_i)(1-\pi_j)k_f+\sigma_i^2\delta_{ij}$. $\sigma_g$ is unused.

**K** scales discrepancy ($K=0$ off; $K=1$ default). Additive/mixture: $K$ multiplies $\sigma_g$. Mixture still gates $f$ by $\pi$ even at $K=0$ (zeros $g$ only). Weighted ignores $\sigma_g$.

Check **$L^{-\omega}f_1$** to add the leading Wegner correction on every selected channel,
$$\Phi=f_0(z)+L^{-\omega}f_1(z),$$
with $\omega=\omega_{\mathrm{exact}}=2$ (not the discrepancy $\omega$ slider).  Recompiles.  The bottom GP panel is then $f_0$ (points can sit off it by the $L^{-\omega}f_1$ piece).  With an additive $a\cdot g$ model the correction replaces that mean; noise inflation still stacks on top.

- **Top-left:** $\Phi_m$ vs $z$ at exact exponents (bars = $s_\sigma\sigma_{\mathrm{MC}}$ or $\sigma_{\mathrm{eff}}$). Marker and error-bar opacity is $|f|/(|f|+|\Phi-f|)$ (smaller $f$ share $\Rightarrow$ more transparent).
- **Heatmaps:** $(T_c,\nu)$ at $\beta=\beta_{\mathrm{exact}}$. Check **β free** to also show $(\nu,\beta)$ at $T_c=T_{c,\mathrm{exact}}$ (so $\beta$ is a free exponent on that slice).
- **Bottom:** posterior of $f(z)$ (mean $\pm 1\sigma$ band) at exact exponents, trained on included points — same style as the AL stepper GP panels.

**Selection (collapse plot):** legend click includes/excludes all points of that `L` and recomputes the heatmap. Click a marker to toggle one point (grey = excluded). Box/lasso-drag to toggle a group. Use **Include all points** to reset.

Use the **dataset**, **model**, and **coupling** dropdowns. Changing dataset or model reloads; coupling recompiles in place.


In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from ising.constants import (
    BETA_EXACT as ISING_BETA_EXACT,
    NU_EXACT as ISING_NU_EXACT,
    OMEGA_EXACT,
    OMEGA_PRIOR_LOWER,
    OMEGA_PRIOR_UPPER,
    TC_EXACT as ISING_TC_EXACT,
)
from ising.datasets import (
    get_dataset as get_ising_dataset,
    list_datasets as list_ising_datasets,
    observables_path as ising_observables_path,
)
from phi4 import fss_io as phi4_fss
from phi4.datasets import (
    get_dataset as get_phi4_dataset,
    list_datasets as list_phi4_datasets,
    observables_path as phi4_observables_path,
)
from ising.discrepancy import (
    DISCREPANCY_KAPPA_INIT,
    DISCREPANCY_KAPPA_PRIOR_LOWER,
    DISCREPANCY_L0_INIT,
    DISCREPANCY_L0_PRIOR_LOWER,
    DISCREPANCY_L0_PRIOR_UPPER,
    DISCREPANCY_P_INIT,
    DISCREPANCY_P_PRIOR_LOWER,
    DISCREPANCY_P_PRIOR_UPPER,
    DISCREPANCY_Q_INIT,
    DISCREPANCY_Q_PRIOR_LOWER,
    DISCREPANCY_Q_PRIOR_UPPER,
    DISCREPANCY_T0_INIT,
    DISCREPANCY_T0_PRIOR_LOWER,
    DISCREPANCY_T0_PRIOR_UPPER,
    discrepancy_amplitude,
    discrepancy_amplitude_fss,
    discrepancy_mix_weight,
    effective_obs_sigma,
)
from ising.fss_likelihood import (
    compile_fss_log_marginal_likelihood_jax,
    default_gp_ell_profile_grid,
    profile_gp_ell,
    profile_joint_with_disc,
)
from ising.gp_jax import (
    DEFAULT_GP_KERNEL,
    discrepancy_f_posterior_predictive,
    gp_posterior_predictive,
)
from ising.jax_config import configure_jax, jax_device_summary
from ising.model_fss import (
    BETA_PRIOR_LOWER,
    BETA_PRIOR_UPPER,
    NU_PRIOR_LOWER,
    NU_PRIOR_UPPER,
)
from ising.model_scaling_nu import (
    TC_PRIOR_LOWER as ISING_TC_PRIOR_LOWER,
    TC_PRIOR_UPPER as ISING_TC_PRIOR_UPPER,
)
from ising.observables import read_observables_for_fss, required_observable_columns
from ising.profile_likelihood import (
    FssProfileConfig,
    Z_DISC_THRESHOLD,
)
from ising.traditional_scaling import magnetization_collapse_arrays

# Active truth / prior window (switched when loading Ising vs φ⁴ datasets).
TC_EXACT = ISING_TC_EXACT
NU_EXACT = ISING_NU_EXACT
BETA_EXACT = ISING_BETA_EXACT
TC_PRIOR_LOWER = ISING_TC_PRIOR_LOWER
TC_PRIOR_UPPER = ISING_TC_PRIOR_UPPER

configure_jax()
print(jax_device_summary())


/Users/reubencohn-gordon/HMCLib-1/.venv/lib/python3.11/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


backend=cpu, devices=[cpu:0]


In [2]:
DEFAULT_FAMILY = "ising"
DEFAULT_DATASET = "harada_square_large_t"
DEFAULT_DISC_FORM = "noise"  # or "additive_gp" / "additive_gp_z_threshold" / "additive_gp_fss"
DEFAULT_COUPLING = "additive"  # or "mixture" / "weighted" (GP forms only)
HEATMAP_N = 35
HEATMAP_Z_FLOOR = -80.0
HEATMAP_ZOOM_LEVELS = (-2.0, -6.0, -20.0, -80.0)
HEATMAP_MIN_ZOOM_CELLS = 4
COLLAPSE_WIDTH = 520
COLLAPSE_HEIGHT = 420
HEATMAP_WIDTH = 480
HEATMAP_HEIGHT = 420
GP_PLOT_N = 250
GP_PLOT_HEIGHT = 360
GP_ELL_INIT = 2.0
GP_ELL_SLIDER_MIN = 0.02
GP_ELL_SLIDER_MAX = 10.0
GP_ELL_PROFILE_N = 17
KAPPA_PROFILE_N = 21
GP_ETA_INIT = 1.0
GP_ETA_SLIDER_MIN = 0.05
GP_ETA_SLIDER_MAX = 5.0
OBS_SIGMA_SCALE_INIT = 1.0
OBS_SIGMA_SCALE_SLIDER_MIN = 0.25
OBS_SIGMA_SCALE_SLIDER_MAX = 50.0
MIN_INCLUDED_POINTS = 4
SIGMA_G_INIT = 10.0
SIGMA_SLIDER_MAX = 20.0
KAPPA_SLIDER_MAX = 100.0

PLOTLY_COLORS = (
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
)
EXCLUDED_COLOR = "#b0b0b0"

DISC_FORM_OPTIONS = (
    ("noise inflation (σ_eff)", "noise"),
    ("additive a·g GP", "additive_gp"),
    ("a=1 if |z|>10 else 0", "additive_gp_z_threshold"),
    ("FSS a = L^{-ω} + κ|t|^{ων}", "additive_gp_fss"),
)

DISC_FORM_LABELS = {
    "noise": "noise inflation",
    "additive_gp": "additive a·g GP",
    "additive_gp_z_threshold": "a=1_{|z|>10} GP",
    "additive_gp_fss": "FSS L^{-ω}+κ|t|^{ων} GP",
}

COUPLING_OPTIONS = (
    ("additive Φ=f+a g", "additive"),
    ("mixture (1-π)f+π g", "mixture"),
    ("weighted (1-π)f", "weighted"),
)

COUPLING_LABELS = {
    "additive": "additive",
    "mixture": "mixture",
    "weighted": "weighted",
}

_ADDITIVE_FORMS = ("additive_gp", "additive_gp_z_threshold", "additive_gp_fss")


def _is_additive_form(form: str | None = None) -> bool:
    f = state["disc_form"] if form is None else form
    return f in _ADDITIVE_FORMS


def _coupling(value: str | None = None) -> str:
    if value is not None:
        return str(value)
    w = globals().get("dropdown_coupling")
    if w is not None:
        return str(w.value)
    stored = state.get("disc_coupling")
    return str(stored) if stored else DEFAULT_COUPLING


def _is_gated_coupling(coupling: str | None = None) -> bool:
    return _coupling(coupling) in ("mixture", "weighted")


def _correction_on() -> bool:
    cfg = state.get("config")
    if cfg is None:
        return False
    return bool(
        getattr(cfg, "correction_m", False)
        or getattr(cfg, "correction_m2", False)
        or getattr(cfg, "correction_m4", False)
        or getattr(cfg, "correction_binder", False)
    )


_CHANNEL_KEYS = ("m", "m2", "m4", "binder")
_DEFAULT_CHANNELS = {"m": True, "m2": True, "m4": False, "binder": False}


def _channel_tag(ch: dict[str, bool] | None = None) -> str:
    labels = {"m": "m", "m2": "m²", "m4": "m⁴", "binder": "U₄"}
    src = _DEFAULT_CHANNELS if ch is None else ch
    parts = [labels[k] for k in _CHANNEL_KEYS if src.get(k)]
    return "+".join(parts) if parts else "none"


def _csv_header_cols(family: str, name: str) -> set[str]:
    path = _observables_path(family, name)
    if not path.is_file():
        return set()
    header = path.read_text().splitlines()[0]
    return {c.strip() for c in header.split(",")}


def _dataset_channels(family: str, name: str) -> dict[str, bool]:
    cols = _csv_header_cols(family, name)

    def _has(*, use_m: bool = False, use_m2: bool = False, use_m4: bool = False, use_binder: bool = False) -> bool:
        needed = required_observable_columns(
            use_m=use_m,
            use_m2=use_m2,
            use_m4=use_m4,
            use_binder=use_binder,
            use_chi=False,
        )
        return needed.issubset(cols)

    return {
        "m": _has(use_m=True),
        "m2": _has(use_m2=True),
        "m4": _has(use_m4=True),
        "binder": _has(use_binder=True),
    }


def _requested_channels() -> dict[str, bool]:
    boxes = {
        "m": globals().get("checkbox_ch_m"),
        "m2": globals().get("checkbox_ch_m2"),
        "m4": globals().get("checkbox_ch_m4"),
        "binder": globals().get("checkbox_ch_binder"),
    }
    if all(b is not None for b in boxes.values()):
        return {k: bool(b.value) for k, b in boxes.items()}
    stored = state.get("channels")
    if isinstance(stored, dict) and stored:
        return {k: bool(stored.get(k, False)) for k in _CHANNEL_KEYS}
    return dict(_DEFAULT_CHANNELS)


def _resolve_channels(available: dict[str, bool]) -> dict[str, bool]:
    wanted = _requested_channels()
    out = {k: bool(wanted.get(k, False) and available.get(k, False)) for k in _CHANNEL_KEYS}
    if not any(out.values()):
        for k in _CHANNEL_KEYS:
            if available.get(k):
                out[k] = True
                break
    return out


def make_config(
    discrepancy_form: str,
    *,
    correction: bool = False,
    channels: dict[str, bool] | None = None,
    coupling: str | None = None,
) -> FssProfileConfig:
    avail = state.get("available_channels") or {k: True for k in _CHANNEL_KEYS}
    ch = _resolve_channels(avail) if channels is None else dict(channels)
    use_m = bool(ch.get("m", False))
    use_m2 = bool(ch.get("m2", False))
    use_m4 = bool(ch.get("m4", False))
    use_binder = bool(ch.get("binder", False))
    coup = "additive"
    if discrepancy_form in _ADDITIVE_FORMS:
        coup = _coupling(coupling)
    return FssProfileConfig(
        use_m=use_m,
        use_m2=use_m2,
        use_m4=use_m4,
        use_binder=use_binder,
        use_chi=False,
        correction_m=bool(correction) and use_m,
        correction_m2=bool(correction) and use_m2,
        correction_m4=bool(correction) and use_m4,
        correction_binder=bool(correction) and use_binder,
        correction_chi=False,
        discrepancy_m=use_m,
        discrepancy_m2=use_m2,
        discrepancy_m4=use_m4,
        discrepancy_binder=use_binder,
        discrepancy_chi=False,
        discrepancy_form=discrepancy_form,  # type: ignore[arg-type]
        discrepancy_coupling=coup,  # type: ignore[arg-type]
        z_disc_threshold=float(Z_DISC_THRESHOLD),
        use_log_m=False,
        omega_fixed=float(OMEGA_EXACT),
        gp_kernel="gaussian",
        # Absolute length scale in z (not factor × data z_span).
        gp_ell=GP_ELL_INIT,
        correction_gp_ell=GP_ELL_INIT,
        gp_eta=GP_ETA_INIT,
        correction_gp_eta=GP_ETA_INIT,
    )


def _form_tag(form: str | None = None) -> str:
    f = state["disc_form"] if form is None else form
    if f == "additive_gp":
        tag = "a·g"
    elif f == "additive_gp_z_threshold":
        tag = "|z|>10"
    elif f == "additive_gp_fss":
        tag = "FSS a"
    else:
        tag = "σ_eff"
    if _is_additive_form(f) and _is_gated_coupling():
        coup = _coupling()
        tag = f"{tag}+{'mix' if coup == 'mixture' else 'w'}"
    if _correction_on():
        tag = f"{tag}+f₁"
    return tag


def _profile_grid(lo: float, hi: float, exact: float, n: int) -> np.ndarray:
    grid = np.linspace(lo, hi, n)
    if lo <= exact <= hi and not np.any(np.isclose(grid, exact, rtol=0.0, atol=1e-10)):
        grid = np.sort(np.append(grid, exact))
    return grid


tc_grid = _profile_grid(TC_PRIOR_LOWER, TC_PRIOR_UPPER, TC_EXACT, HEATMAP_N)
nu_grid = _profile_grid(NU_PRIOR_LOWER, NU_PRIOR_UPPER, NU_EXACT, HEATMAP_N)
beta_grid = _profile_grid(BETA_PRIOR_LOWER, BETA_PRIOR_UPPER, BETA_EXACT, HEATMAP_N)


def _observables_path(family: str, name: str) -> Path:
    if family == "ising":
        return ising_observables_path(name)
    if family == "phi4":
        return phi4_observables_path(name)
    raise ValueError(f"Unknown dataset family {family!r}")


def _get_dataset(family: str, name: str):
    if family == "ising":
        return get_ising_dataset(name)
    if family == "phi4":
        return get_phi4_dataset(name)
    raise ValueError(f"Unknown dataset family {family!r}")


def _dataset_has_any_channel(family: str, name: str) -> bool:
    return any(_dataset_channels(family, name).values())


def _set_truth_for_family(family: str) -> None:
    """Switch active exact exponents / Tc prior window (mutates module globals)."""
    global TC_EXACT, NU_EXACT, BETA_EXACT, TC_PRIOR_LOWER, TC_PRIOR_UPPER
    global tc_grid, nu_grid, beta_grid
    if family == "ising":
        TC_EXACT = ISING_TC_EXACT
        NU_EXACT = ISING_NU_EXACT
        BETA_EXACT = ISING_BETA_EXACT
        TC_PRIOR_LOWER = ISING_TC_PRIOR_LOWER
        TC_PRIOR_UPPER = ISING_TC_PRIOR_UPPER
    elif family == "phi4":
        TC_EXACT, NU_EXACT, BETA_EXACT = phi4_fss.truth_exponents()
        TC_PRIOR_LOWER = phi4_fss.TC_PRIOR_LOWER
        TC_PRIOR_UPPER = phi4_fss.TC_PRIOR_UPPER
    else:
        raise ValueError(f"Unknown dataset family {family!r}")
    tc_grid = _profile_grid(TC_PRIOR_LOWER, TC_PRIOR_UPPER, TC_EXACT, HEATMAP_N)
    nu_grid = _profile_grid(NU_PRIOR_LOWER, NU_PRIOR_UPPER, NU_EXACT, HEATMAP_N)
    beta_grid = _profile_grid(BETA_PRIOR_LOWER, BETA_PRIOR_UPPER, BETA_EXACT, HEATMAP_N)


AVAILABLE_DATASETS: list[tuple[str, tuple[str, str]]] = []
for _name in list_ising_datasets():
    if _dataset_has_any_channel("ising", _name):
        AVAILABLE_DATASETS.append((f"ising/{_name}", ("ising", _name)))
for _name in list_phi4_datasets():
    if _dataset_has_any_channel("phi4", _name):
        AVAILABLE_DATASETS.append((f"phi4/{_name}", ("phi4", _name)))

_default_key = (DEFAULT_FAMILY, DEFAULT_DATASET)
if not any(val == _default_key for _, val in AVAILABLE_DATASETS):
    raise FileNotFoundError(
        f"Default dataset {DEFAULT_FAMILY}/{DEFAULT_DATASET!r} CSV missing; "
        f"available: {[lab for lab, _ in AVAILABLE_DATASETS]}"
    )

state: dict = {
    "family": None,
    "dataset": None,
    "disc_form": None,
    "disc_coupling": DEFAULT_COUPLING,
    "correction": False,
    "channels": dict(_DEFAULT_CHANNELS),
    "available_channels": {k: True for k in _CHANNEL_KEYS},
    "config": None,
    "df_full": None,
    "df": None,
    "included": None,
    "compiled": None,
    "z0": None,
    "phi0": None,
    "sigma_phi_mc": None,
    "L_arr": None,
    "t0_arr": None,
    "T_arr": None,
    "L_values": (),
    "Y_LO": 0.0,
    "Y_HI": 1.0,
}


def _active_df():
    mask = np.asarray(state["included"], dtype=bool)
    return state["df_full"].iloc[np.flatnonzero(mask)].reset_index(drop=True)


def _recompile_active(*, quiet: bool = True) -> bool:
    """Recompile JAX likelihood on currently included points. Returns False if too few."""
    mask = np.asarray(state["included"], dtype=bool)
    n_in = int(mask.sum())
    if n_in < MIN_INCLUDED_POINTS:
        return False
    df_active = state["df_full"].iloc[np.flatnonzero(mask)].reset_index(drop=True)
    compiled = compile_fss_log_marginal_likelihood_jax(df_active, state["config"])
    state["df"] = df_active
    state["compiled"] = compiled
    if not quiet:
        print(f"Recompiled on {n_in}/{len(mask)} included points")
    return True


def _correction_requested() -> bool:
    w = globals().get("checkbox_correction")
    if w is not None:
        return bool(w.value)
    return bool(state.get("correction", False))


def load_dataset(
    name: str,
    *,
    family: str | None = None,
    discrepancy_form: str | None = None,
    coupling: str | None = None,
    correction: bool | None = None,
    quiet: bool = False,
) -> None:
    """Load observables, compile likelihood, and refresh collapse arrays."""
    fam = DEFAULT_FAMILY if family is None else str(family)
    form = DEFAULT_DISC_FORM if discrepancy_form is None else str(discrepancy_form)
    coup = DEFAULT_COUPLING if coupling is None else str(coupling)
    use_corr = _correction_requested() if correction is None else bool(correction)
    _set_truth_for_family(fam)
    avail = _dataset_channels(fam, name)
    ch = _resolve_channels(avail)
    config = make_config(form, correction=use_corr, channels=ch, coupling=coup)
    data_path = _observables_path(fam, name)
    if fam == "phi4":
        df = phi4_fss.load_observables(name, path=data_path)
    else:
        df = read_observables_for_fss(
            data_path,
            use_m=avail["m"],
            use_m2=avail["m2"],
            use_m4=avail["m4"],
            use_binder=avail["binder"],
            use_chi=False,
        )
    included = np.ones(len(df), dtype=bool)
    if "magnetization" not in df.columns:
        raise ValueError(
            f"{fam}/{name} has no magnetization column; the collapse plot needs m "
            f"(available={ {k: v for k, v in avail.items() if v} })"
        )
    z0, phi0, sigma_phi_mc, L_arr = magnetization_collapse_arrays(
        df, TC_EXACT, NU_EXACT, BETA_EXACT
    )
    T_arr = df["T"].to_numpy(dtype=np.float64)
    t0_arr = (T_arr - TC_EXACT) / TC_EXACT
    L_values = tuple(sorted(int(v) for v in np.unique(L_arr)))
    y_pad = 0.08 * float(np.ptp(phi0) + 1e-12)
    state.update(
        family=fam,
        dataset=name,
        disc_form=form,
        disc_coupling=coup,
        correction=use_corr,
        channels=ch,
        available_channels=avail,
        config=config,
        df_full=df,
        included=included,
        z0=z0,
        phi0=phi0,
        sigma_phi_mc=sigma_phi_mc,
        L_arr=L_arr,
        t0_arr=t0_arr,
        T_arr=T_arr,
        L_values=L_values,
        Y_LO=float(np.min(phi0 - sigma_phi_mc)) - y_pad,
        Y_HI=float(np.max(phi0 + sigma_phi_mc)) + y_pad,
    )
    if not _recompile_active(quiet=True):
        raise RuntimeError(f"Dataset {fam}/{name!r} has fewer than {MIN_INCLUDED_POINTS} points")
    if not quiet:
        spec = _get_dataset(fam, name)
        form_label = DISC_FORM_LABELS.get(form, form)
        print(
            f"Loaded {fam}/{name}: {len(df)} points from {data_path}\n"
            f"  {spec.description}\n"
            f"Model: {form_label}; coupling={coup}; channels={_channel_tag(ch)}; "
            f"correction={'on' if use_corr else 'off'} (ω={OMEGA_EXACT:g}); "
            f"gp_ell={state['compiled'].gp_scales.base_gp_ell:.4g}, "
            f"gp_eta={state['compiled'].gp_scales.base_gp_eta:.4g}; "
            f"L={L_values}; heatmap {tc_grid.size}×{nu_grid.size} (β {beta_grid.size}); "
            f"Tc_exact={TC_EXACT:.5g}"
        )


load_dataset(DEFAULT_DATASET, family=DEFAULT_FAMILY, discrepancy_form=DEFAULT_DISC_FORM)


Loaded ising/harada_square_large_t: 151 points from /Users/reubencohn-gordon/HMCLib-1/ising/data/observables_harada_square_large_t.csv
  Harada square Binder grid (L=64/128/256) plus small-L×large-|t| points (L=8–32, T∈[1.70,8]) to expose FSS collapse failure away from the joint t→0, L→∞ limit.
Model: noise inflation; coupling=additive; channels=m+m²; correction=off (ω=2); gp_ell=2, gp_eta=1; L=(8, 12, 16, 24, 32, 64, 128, 256); heatmap 36×35 (β 36); Tc_exact=2.2692


In [3]:
def _disc_params() -> dict[str, float]:
    return {
        "t0": float(slider_t0.value),
        "L0": float(slider_L0.value),
        "p": float(slider_p.value),
        "q": float(slider_q.value),
        "omega": float(slider_omega.value),
        "kappa": float(slider_kappa.value),
        "sigma_model": float(slider_sigma.value),
        "K": float(slider_K.value),
        "gp_ell": float(slider_gp_ell.value),
        "disc_gp_ell": float(slider_disc_ell.value),
        "gp_eta": float(slider_gp_eta.value),
        "obs_sigma_scale": float(slider_obs_sigma.value),
    }


def _disc_profile_slots(params: dict[str, float]) -> tuple[float, float, float, float]:
    """Map sliders onto evaluate_with_gp (t0, L0, p, q).

    additive_gp_fss reuses t0=κ, q=ω (L0 and p unused).
    """
    if state["disc_form"] == "additive_gp_fss":
        return (
            float(params["kappa"]),
            1.0,
            2.0,
            float(params["omega"]),
        )
    return (
        float(params["t0"]),
        float(params["L0"]),
        float(params["p"]),
        float(params["q"]),
    )


def _n_included() -> int:
    return int(np.asarray(state["included"], dtype=bool).sum())


def _inclusion_status_text() -> str:
    n_tot = len(state["included"])
    n_in = _n_included()
    return f"included {n_in}/{n_tot}"


def _inclusion_status_html() -> str:
    n_tot = len(state["included"])
    n_in = _n_included()
    return f"included <b>{n_in}</b>/{n_tot}"


def _scaled_sigma_phi_mc(params: dict[str, float]) -> np.ndarray:
    """MC σ_Φ multiplied by the uniform observation-noise scale s_σ."""
    return np.asarray(state["sigma_phi_mc"], dtype=np.float64) * float(
        params.get("obs_sigma_scale", 1.0)
    )


def _sigma_eff_phi(params: dict[str, float]) -> tuple[np.ndarray, np.ndarray]:
    """Return (K·a_vis, sigma_plot). Additive forms: bars are s_σ·σ_MC.

    K scales the discrepancy. For z-threshold, a_vis = 1_{|z|>thr} at exact
    collapse z (display only; heatmap recomputes a from z(Tc,ν)).
    For FSS form, a_vis is collapsed a = L^{-ω} + κ |t|^{ω ν} at exact ν.
    """
    sigma_mc = _scaled_sigma_phi_mc(params)
    if state["disc_form"] == "additive_gp_z_threshold":
        thr = float(getattr(state["config"], "z_disc_threshold", Z_DISC_THRESHOLD))
        a = (np.abs(np.asarray(state["z0"], dtype=np.float64)) > thr).astype(
            np.float64
        )
        Ka = float(params["K"]) * a
        return Ka, sigma_mc

    if state["disc_form"] == "additive_gp_fss":
        a = discrepancy_amplitude_fss(
            state["t0_arr"],
            state["L_arr"],
            omega=float(params["omega"]),
            kappa=float(params["kappa"]),
            nu=NU_EXACT,
        )
        Ka = float(params["K"]) * a
        return Ka, sigma_mc

    a = discrepancy_amplitude(
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
    )
    Ka = float(params["K"]) * a
    if state["disc_form"] == "additive_gp":
        return Ka, sigma_mc
    sigma_eff = effective_obs_sigma(
        sigma_mc,
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
        sigma_model=params["sigma_model"] * float(params["K"]),
    )
    return Ka, sigma_eff


def _window_amplitude(params: dict[str, float]) -> np.ndarray:
    """Window a(t,L) used to form the mixture/weighted gate π (not Φ-scaled)."""
    form = state["disc_form"]
    if form == "additive_gp_z_threshold":
        thr = float(getattr(state["config"], "z_disc_threshold", Z_DISC_THRESHOLD))
        return (np.abs(np.asarray(state["z0"], dtype=np.float64)) > thr).astype(
            np.float64
        )
    if form == "additive_gp_fss":
        return discrepancy_amplitude_fss(
            state["t0_arr"],
            state["L_arr"],
            omega=float(params["omega"]),
            kappa=float(params["kappa"]),
            nu=NU_EXACT,
        )
    return discrepancy_amplitude(
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
    )


def _gate_pi(params: dict[str, float]) -> np.ndarray:
    """Unit gate π for mixture/weighted; ones for additive/noise."""
    if not (_is_additive_form() and _is_gated_coupling()):
        return np.ones_like(np.asarray(state["z0"], dtype=np.float64))
    a = _window_amplitude(params)
    squash = state["disc_form"] != "additive_gp_z_threshold"
    return np.asarray(discrepancy_mix_weight(a, squash=squash), dtype=np.float64)



def _f_hat_phi_m(params: dict[str, float]) -> np.ndarray:
    """GP posterior mean of universal f at exact-(Tc,ν,β) collapse points.

    Trained on currently included points only (same mask as the heatmap LML).
    Noise model: latent GP with σ_eff. Additive GP: E[f|y] from
    Φ = f + (a L^{β/ν}) g + ε (K absorbed into σ_g, matching the heatmap).
    """
    z = np.asarray(state["z0"], dtype=np.float64)
    mean, _ = _f_posterior(params, z)
    return mean


def _f_posterior(
    params: dict[str, float], z_new: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Posterior mean and std of universal f at ``z_new``."""
    included = np.asarray(state["included"], dtype=bool)
    z = np.asarray(state["z0"], dtype=np.float64)
    phi = np.asarray(state["phi0"], dtype=np.float64)
    z_new = np.asarray(z_new, dtype=np.float64).ravel()
    n_new = int(z_new.size)
    empty = (
        np.full(n_new, np.nan, dtype=np.float64),
        np.full(n_new, np.nan, dtype=np.float64),
    )
    if int(included.sum()) < 2:
        return empty

    a_full = discrepancy_amplitude(
        state["t0_arr"],
        state["L_arr"],
        t0=params["t0"],
        L0=params["L0"],
        p=params["p"],
        q=params["q"],
    )
    sigma_mc = _scaled_sigma_phi_mc(params)
    ell = float(params.get("gp_ell", GP_ELL_INIT))
    eta = float(params.get("gp_eta", GP_ETA_INIT))
    kernel = getattr(state["config"], "gp_kernel", None) or DEFAULT_GP_KERNEL

    z_tr = z[included]
    y_tr = phi[included]
    if _correction_on():
        # Likelihood: Φ = f0(z) + L^{-ω} f1(z) (ω = ω_exact).  Return E[f0|y].
        # Additive a·g is not used for the mean while correction is on;
        # noise inflation still applies for the noise-form discrepancy.
        L_arr = np.asarray(state["L_arr"], dtype=np.float64)
        a_tr = L_arr[included] ** (-float(OMEGA_EXACT))
        scales = state["compiled"].gp_scales
        if state["disc_form"] == "noise":
            _, sigma_plot = _sigma_eff_phi(params)
            sigma_tr = np.asarray(sigma_plot, dtype=np.float64)[included]
        else:
            sigma_tr = sigma_mc[included]
        mean, std = discrepancy_f_posterior_predictive(
            z_tr,
            y_tr,
            sigma_tr,
            a_tr,
            z_new,
            gp_ell=ell,
            gp_eta=eta,
            disc_gp_ell=float(scales.correction_gp_ell),
            disc_gp_eta=float(scales.correction_gp_eta),
            kernel=kernel,
        )
    elif _is_additive_form():
        # Additive: a is the collapsed g-coefficient (Φ-space).
        # Mixture/weighted: a is the unit gate π from the window amplitude.
        coup = _coupling()
        if _is_gated_coupling(coup):
            a_for_gp = _gate_pi(params)
        elif state["disc_form"] == "additive_gp":
            L_arr = np.asarray(state["L_arr"], dtype=np.float64)
            a_for_gp = a_full * (L_arr ** (BETA_EXACT / NU_EXACT))
        else:
            a_for_gp = _window_amplitude(params)
        mean, std = discrepancy_f_posterior_predictive(
            z_tr,
            y_tr,
            sigma_mc[included],
            a_for_gp[included],
            z_new,
            gp_ell=ell,
            gp_eta=eta,
            disc_gp_ell=float(params.get("disc_gp_ell", ell)),
            disc_gp_eta=float(params["sigma_model"]) * float(params["K"]),
            kernel=kernel,
            coupling=coup,
        )
    else:
        sigma_eff = effective_obs_sigma(
            sigma_mc,
            state["t0_arr"],
            state["L_arr"],
            t0=params["t0"],
            L0=params["L0"],
            p=params["p"],
            q=params["q"],
            sigma_model=params["sigma_model"] * float(params["K"]),
        )
        mean, std = gp_posterior_predictive(
            z_tr,
            y_tr,
            sigma_eff[included],
            z_new,
            length_scale=ell,
            amplitude=eta,
            kernel=kernel,
        )
    return (
        np.asarray(mean, dtype=np.float64),
        np.asarray(std, dtype=np.float64),
    )


def _heatmap_axis_ranges(
    z: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    x_mle: float,
    y_mle: float,
    x_exact: float,
    y_exact: float,
    dlog_exact: float,
) -> tuple[list[float], list[float], float, int, bool]:
    x_span = float(x_grid[-1] - x_grid[0])
    y_span = float(y_grid[-1] - y_grid[0])
    d_x = float(np.min(np.diff(x_grid)))
    d_y = float(np.min(np.diff(y_grid)))

    level_used = HEATMAP_Z_FLOOR
    n_cells = 0
    mask = None
    for level in HEATMAP_ZOOM_LEVELS:
        cand = z > level
        n = int(np.count_nonzero(cand))
        if n >= HEATMAP_MIN_ZOOM_CELLS:
            mask = cand
            level_used = float(level)
            n_cells = n
            break
        if n > n_cells:
            mask = cand
            level_used = float(level)
            n_cells = n

    if mask is not None and n_cells >= 1:
        jj, ii = np.where(mask)
        x_lo = float(x_grid[ii.min()])
        x_hi = float(x_grid[ii.max()])
        y_lo = float(y_grid[jj.min()])
        y_hi = float(y_grid[jj.max()])
    else:
        x_lo = x_hi = x_mle
        y_lo = y_hi = y_mle

    x_lo = min(x_lo, x_mle)
    x_hi = max(x_hi, x_mle)
    y_lo = min(y_lo, y_mle)
    y_hi = max(y_hi, y_mle)
    include_exact = bool(np.isfinite(dlog_exact) and dlog_exact > HEATMAP_Z_FLOOR)
    if include_exact:
        x_lo = min(x_lo, x_exact)
        x_hi = max(x_hi, x_exact)
        y_lo = min(y_lo, y_exact)
        y_hi = max(y_hi, y_exact)

    pad_x = max(2.5 * d_x, 0.04 * x_span)
    pad_y = max(2.5 * d_y, 0.04 * y_span)
    min_half_x = 0.06 * x_span
    min_half_y = 0.06 * y_span
    x_c = 0.5 * (x_lo + x_hi)
    y_c = 0.5 * (y_lo + y_hi)
    half_x = max(0.5 * (x_hi - x_lo) + pad_x, min_half_x)
    half_y = max(0.5 * (y_hi - y_lo) + pad_y, min_half_y)

    x_range = [
        max(float(x_grid[0]), x_c - half_x),
        min(float(x_grid[-1]), x_c + half_x),
    ]
    y_range = [
        max(float(y_grid[0]), y_c - half_y),
        min(float(y_grid[-1]), y_c + half_y),
    ]
    exact_in_view = include_exact and (
        x_range[0] <= x_exact <= x_range[1] and y_range[0] <= y_exact <= y_range[1]
    )
    return x_range, y_range, level_used, n_cells, exact_in_view


def _f_share(
    phi: np.ndarray,
    f_hat: np.ndarray,
    w_f: np.ndarray | float | None = None,
) -> np.ndarray:
    """Relative contribution of the f term: |w_f f| / (|w_f f| + |Φ − w_f f|)."""
    phi = np.asarray(phi, dtype=np.float64)
    f = np.asarray(f_hat, dtype=np.float64)
    if w_f is None:
        f_term = f
    else:
        f_term = np.asarray(w_f, dtype=np.float64) * f
    den = np.abs(f_term) + np.abs(phi - f_term)
    share = np.ones(phi.shape, dtype=np.float64)
    ok = np.isfinite(f_term) & np.isfinite(den) & (den > 0.0)
    share[ok] = np.abs(f_term[ok]) / den[ok]
    return share


def _collapse_share_and_pi(
    params: dict[str, float], f_hat: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    pi = _gate_pi(params)
    gated = _is_additive_form() and _is_gated_coupling()
    w_f = (1.0 - pi) if gated else None
    return _f_share(state["phi0"], f_hat, w_f=w_f), pi


def _color_alpha(color: str, opacity: float) -> str:
    h = str(color).lstrip("#")
    if len(h) == 3:
        h = "".join(ch * 2 for ch in h)
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{float(opacity):.3f})"


def _errorbar_shapes_for_L(
    z: np.ndarray,
    phi: np.ndarray,
    sigma: np.ndarray,
    opacity: list[float],
    colors: list[str],
) -> list[dict]:
    """Vertical error bars with the same per-point alpha as the markers."""
    shapes: list[dict] = []
    for zk, yk, ek, op, ck in zip(z, phi, sigma, opacity, colors, strict=True):
        shapes.append(
            dict(
                type="line",
                x0=float(zk),
                x1=float(zk),
                y0=float(yk - ek),
                y1=float(yk + ek),
                xref="x",
                yref="y",
                layer="below",
                line=dict(color=_color_alpha(ck, op), width=1.0),
            )
        )
    return shapes


def _marker_style_for_L(
    Lval: int,
    order: np.ndarray,
    f_share: np.ndarray,
    *,
    color: str,
) -> tuple[list[float], list[float], list[str]]:
    """Return (sizes, opacities, colors) for one L-trace in display order.

    Size is constant. Opacity is the f-share of Φ (|f| / (|f|+|Φ−f|));
    excluded points stay grey and faint.
    """
    included = np.asarray(state["included"], dtype=bool)
    L_arr = state["L_arr"]
    mask = L_arr == Lval
    idxs = np.flatnonzero(mask)[order]
    share_L = f_share[mask][order]
    n = int(idxs.size)
    size = [9.0] * n
    opacity = []
    colors = []
    for j, s_j in zip(idxs, share_L):
        if included[j]:
            colors.append(color)
            opacity.append(float(np.clip(s_j, 0.12, 0.95)))
        else:
            colors.append(EXCLUDED_COLOR)
            opacity.append(0.25)
    return size, opacity, colors


def _build_collapse_figure() -> go.FigureWidget:
    params = {
        "t0": DISCREPANCY_T0_INIT,
        "L0": DISCREPANCY_L0_INIT,
        "p": DISCREPANCY_P_INIT,
        "q": DISCREPANCY_Q_INIT,
        "omega": float(OMEGA_EXACT),
        "kappa": DISCREPANCY_KAPPA_INIT,
        "sigma_model": SIGMA_G_INIT,
        "K": 1.0,
        "gp_ell": GP_ELL_INIT,
        "disc_gp_ell": GP_ELL_INIT,
        "gp_eta": GP_ETA_INIT,
        "obs_sigma_scale": OBS_SIGMA_SCALE_INIT,
    }
    a, sigma_plot = _sigma_eff_phi(params)
    f_hat = _f_hat_phi_m(params)
    f_share, pi = _collapse_share_and_pi(params, f_hat)
    z0 = state["z0"]
    phi0 = state["phi0"]
    L_arr = state["L_arr"]
    T_arr = state["T_arr"]
    L_values = state["L_values"]
    Y_LO = state["Y_LO"]
    Y_HI = state["Y_HI"]
    included = np.asarray(state["included"], dtype=bool)
    traces = []
    err_shapes: list[dict] = []
    for i, Lval in enumerate(L_values):
        mask = L_arr == Lval
        order = np.argsort(z0[mask])
        idxs = np.flatnonzero(mask)[order]
        color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
        size, opacity, colors = _marker_style_for_L(
            Lval, order, f_share, color=color
        )
        n_in_L = int(included[mask].sum())
        visible = True if n_in_L > 0 else "legendonly"
        z_L = z0[mask][order]
        phi_L = phi0[mask][order]
        sig_L = sigma_plot[mask][order]
        if n_in_L > 0:
            err_shapes.extend(
                _errorbar_shapes_for_L(z_L, phi_L, sig_L, opacity, colors)
            )
        traces.append(
            go.Scatter(
                x=z_L.tolist(),
                y=phi_L.tolist(),
                customdata=np.column_stack(
                    [
                        idxs,
                        T_arr[mask][order],
                        a[mask][order],
                        f_hat[mask][order],
                        (phi_L - f_hat[mask][order]),
                        f_share[mask][order],
                        pi[mask][order],
                    ]
                ).tolist(),
                error_y=dict(visible=False),
                mode="markers",
                marker=dict(size=size, color=colors, opacity=opacity),
                name=f"L={Lval}",
                visible=visible,
                hovertemplate=(
                    f"idx=%{{customdata[0]:.0f}}<br>L={Lval}"
                    "<br>T=%{customdata[1]:.4f}<br>z=%{x:.3g}<br>Φ=%{y:.4f}"
                    "<br>f=%{customdata[3]:.4f}<br>Φ−f=%{customdata[4]:.4f}"
                    "<br>|f|/(|f|+|Φ−f|)=%{customdata[5]:.2f}"
                    "<br>a=%{customdata[2]:.3g}<br>π=%{customdata[6]:.3g}<extra></extra>"
                ),
            )
        )
    traces.append(
        go.Scatter(
            x=[0.0, 0.0],
            y=[Y_LO, Y_HI],
            mode="lines",
            line=dict(color="gray", width=1, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        )
    )
    bar_label = "σ_MC" if _is_additive_form() else "σ_eff"
    fig = go.FigureWidget(
        data=traces,
        layout=go.Layout(
            title=dict(
                text=(
                    f"m collapse (LML: {_channel_tag()}) [{state['dataset']}] ({bar_label} bars) · "
                    f"{_inclusion_status_text()}"
                ),
                font=dict(size=13),
            ),
            xaxis_title="z = t L^(1/ν)",
            yaxis_title="Φ_m = |m| L^(β/ν)",
            yaxis=dict(range=[Y_LO, Y_HI]),
            width=COLLAPSE_WIDTH,
            height=COLLAPSE_HEIGHT,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(
                font=dict(size=9),
                yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99,
                itemclick="toggle",
                itemdoubleclick="toggleothers",
            ),
            # Box/lasso select toggles inclusion of selected points.
            dragmode="select",
            clickmode="event+select",
            selectdirection="any",
            hovermode="closest",
            shapes=err_shapes,
        ),
    )
    _bind_collapse_interactions(fig)
    return fig


def _build_joint_heatmap_figure(
    *,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    x_exact: float,
    y_exact: float,
    x_label: str,
    y_label: str,
    title: str,
    hover_x: str,
    hover_y: str,
) -> go.FigureWidget:
    z0 = np.full((y_grid.size, x_grid.size), HEATMAP_Z_FLOOR, dtype=np.float64).tolist()
    x_list = np.asarray(x_grid, dtype=np.float64).tolist()
    y_list = np.asarray(y_grid, dtype=np.float64).tolist()
    return go.FigureWidget(
        data=[
            go.Heatmap(
                x=x_list,
                y=y_list,
                z=z0,
                zmin=HEATMAP_Z_FLOOR,
                zmax=0.0,
                colorscale="Viridis",
                colorbar=dict(title="ΔlogL", thickness=14, len=0.8),
                hovertemplate=(
                    f"{hover_x}=%{{x:.5f}}<br>{hover_y}=%{{y:.4f}}"
                    "<br>ΔlogL=%{z:.2f}<extra></extra>"
                ),
            ),
            go.Contour(
                x=x_list,
                y=y_list,
                z=z0,
                showscale=False,
                contours=dict(start=-20, end=0, size=2, coloring="lines"),
                line=dict(width=0.6, color="rgba(255,255,255,0.35)"),
                hoverinfo="skip",
            ),
            go.Scatter(
                x=[x_exact],
                y=[y_exact],
                mode="markers",
                marker=dict(
                    symbol="circle",
                    size=11,
                    color="white",
                    line=dict(width=1.2, color="black"),
                ),
                name="exact",
                hovertemplate=(
                    f"exact {hover_x}={x_exact:.5g}, {hover_y}={y_exact:g}<extra></extra>"
                ),
            ),
            go.Scatter(
                x=[x_exact],
                y=[y_exact],
                mode="markers",
                marker=dict(
                    symbol="x",
                    size=12,
                    color="#c44e52",
                    line=dict(width=1.5, color="white"),
                ),
                name="grid MLE",
                hovertemplate=(
                    f"MLE {hover_x}=%{{x:.5f}}, {hover_y}=%{{y:.4f}}<extra></extra>"
                ),
            ),
        ],
        layout=go.Layout(
            title=dict(text=title, font=dict(size=13)),
            xaxis_title=x_label,
            yaxis_title=y_label,
            xaxis=dict(range=[float(x_grid[0]), float(x_grid[-1])]),
            yaxis=dict(range=[float(y_grid[0]), float(y_grid[-1])]),
            width=HEATMAP_WIDTH,
            height=HEATMAP_HEIGHT,
            template="plotly_white",
            margin=dict(l=48, r=72, t=48, b=48),
            legend=dict(font=dict(size=9), yanchor="top", y=0.99, xanchor="left", x=0.01),
        ),
    )


def _build_heatmap_figure() -> go.FigureWidget:
    return _build_joint_heatmap_figure(
        x_grid=tc_grid,
        y_grid=nu_grid,
        x_exact=TC_EXACT,
        y_exact=NU_EXACT,
        x_label="Tc",
        y_label="ν",
        title="(Tc, ν) log-marginal (β fixed)",
        hover_x="Tc",
        hover_y="ν",
    )


def _build_beta_heatmap_figure() -> go.FigureWidget:
    return _build_joint_heatmap_figure(
        x_grid=nu_grid,
        y_grid=beta_grid,
        x_exact=NU_EXACT,
        y_exact=BETA_EXACT,
        x_label="ν",
        y_label="β",
        title="(ν, β) log-marginal (Tc fixed)",
        hover_x="ν",
        hover_y="β",
    )


def _point_hovertemplate(Lval: int) -> str:
    return (
        f"idx=%{{customdata[0]:.0f}}<br>L={Lval}"
        "<br>T=%{customdata[1]:.4f}<br>z=%{x:.3g}<br>Φ=%{y:.4f}"
        "<br>f=%{customdata[3]:.4f}<br>Φ−f=%{customdata[4]:.4f}"
        "<br>|f|/(|f|+|Φ−f|)=%{customdata[5]:.2f}"
        "<br>a=%{customdata[2]:.3g}<br>π=%{customdata[6]:.3g}<extra></extra>"
    )


def _gp_plot_width() -> int:
    return COLLAPSE_WIDTH + HEATMAP_WIDTH


def _z_grid_for_gp() -> np.ndarray:
    z = np.asarray(state["z0"], dtype=np.float64)
    z = z[np.isfinite(z)]
    if z.size == 0:
        return np.linspace(-1.0, 1.0, GP_PLOT_N)
    z_lo = float(np.min(z))
    z_hi = float(np.max(z))
    pad = 0.06 * (z_hi - z_lo + 1e-12)
    return np.linspace(z_lo - pad, z_hi + pad, GP_PLOT_N)


def _build_gp_figure() -> go.FigureWidget:
    """Universal f(z) posterior: mean ±1σ band + collapsed Φ_m by L."""
    L_values = state["L_values"]
    traces: list = [
        go.Scatter(
            x=[],
            y=[],
            mode="lines",
            line=dict(width=0, color="#222222"),
            hoverinfo="skip",
            showlegend=False,
            name="band_lo",
        ),
        go.Scatter(
            x=[],
            y=[],
            mode="lines",
            line=dict(width=0, color="#222222"),
            fill="tonexty",
            fillcolor="rgba(34,34,34,0.18)",
            hoverinfo="skip",
            showlegend=False,
            name="band_hi",
        ),
        go.Scatter(
            x=[],
            y=[],
            mode="lines",
            line=dict(color="#222222", width=2.0),
            name="f(z)",
            hovertemplate="z=%{x:.3g}<br>f=%{y:.4f}<extra></extra>",
        ),
        go.Scatter(
            x=[0.0, 0.0],
            y=[0.0, 1.0],
            mode="lines",
            line=dict(color="gray", width=1, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
            name="z=0",
        ),
    ]
    for i, Lval in enumerate(L_values):
        color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
        traces.append(
            go.Scatter(
                x=[],
                y=[],
                mode="markers",
                marker=dict(size=9, color=color, opacity=0.85),
                name=f"L={Lval}",
                error_y=dict(visible=False),
                hovertemplate=_point_hovertemplate(Lval),
            )
        )
    return go.FigureWidget(
        data=traces,
        layout=go.Layout(
            title=dict(text="f(z) GP posterior ±1σ (opacity = f-share)", font=dict(size=13)),
            xaxis_title="z = t L^(1/ν)",
            yaxis_title="f(z)",
            width=_gp_plot_width(),
            height=GP_PLOT_HEIGHT,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(
                font=dict(size=9),
                yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99,
            ),
            hovermode="closest",
            shapes=[],
        ),
    )


def _build_ell_profile_figure() -> go.FigureWidget:
    """ΔlogL vs ℓ_f at exact (Tc, ν, β)."""
    return go.FigureWidget(
        data=[
            go.Scatter(
                x=[],
                y=[],
                mode="lines+markers",
                line=dict(color="#1f77b4", width=2),
                marker=dict(size=6),
                name="ΔlogL",
                hovertemplate="ℓ_f=%{x:.3g}<br>ΔlogL=%{y:.2f}<extra></extra>",
            ),
            go.Scatter(
                x=[],
                y=[],
                mode="markers",
                marker=dict(size=11, color="#d62728", symbol="x"),
                name="slider",
                hovertemplate="slider ℓ_f=%{x:.3g}<extra></extra>",
            ),
            go.Scatter(
                x=[],
                y=[],
                mode="markers",
                marker=dict(size=11, color="#2ca02c", symbol="diamond"),
                name="ℓ*",
                hovertemplate="ℓ*=%{x:.3g}<extra></extra>",
            ),
        ],
        layout=go.Layout(
            title=dict(text="profile ℓ_f at exact (Tc,ν,β)", font=dict(size=13)),
            xaxis=dict(
                title="ℓ_f (z-units, log)",
                type="log",
                tickvals=[0.02, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0],
                ticktext=["0.02", "0.05", "0.1", "0.2", "0.5", "1", "2", "5", "10"],
            ),
            yaxis_title="ΔlogL",
            width=HEATMAP_WIDTH,
            height=GP_PLOT_HEIGHT,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(font=dict(size=9), yanchor="top", y=0.99, xanchor="right", x=0.99),
            hovermode="closest",
        ),
    )


def _update_gp(params: dict[str, float]) -> None:
    """Refresh the f(z) mean ±1σ panel (AL-stepper style)."""
    z0 = np.asarray(state["z0"], dtype=np.float64)
    phi0 = np.asarray(state["phi0"], dtype=np.float64)
    L_arr = np.asarray(state["L_arr"], dtype=np.float64)
    T_arr = np.asarray(state["T_arr"], dtype=np.float64)
    L_values = state["L_values"]
    included = np.asarray(state["included"], dtype=bool)
    a, sigma_plot = _sigma_eff_phi(params)
    f_hat = _f_hat_phi_m(params)
    f_share, pi = _collapse_share_and_pi(params, f_hat)
    z_grid = _z_grid_for_gp()
    mean, std = _f_posterior(params, z_grid)
    ok = np.isfinite(mean) & np.isfinite(std)
    if np.any(ok):
        y_lo = mean[ok] - std[ok]
        y_hi = mean[ok] + std[ok]
        x_band = z_grid[ok]
        y_mean = mean[ok]
    else:
        y_lo = y_hi = y_mean = np.array([], dtype=np.float64)
        x_band = np.array([], dtype=np.float64)

    y_vals = [phi0]
    if y_lo.size:
        y_vals.extend([y_lo, y_hi])
    y_all = np.concatenate([np.ravel(v) for v in y_vals])
    y_all = y_all[np.isfinite(y_all)]
    if y_all.size:
        y_pad = 0.08 * float(np.ptp(y_all) + 1e-12)
        y_range = [float(np.min(y_all)) - y_pad, float(np.max(y_all)) + y_pad]
    else:
        y_range = [0.0, 1.0]

    n_l = len(L_values)
    # FigureWidget may lag a rebuild; skip if L-trace count doesn't match.
    if len(gp_fig.data) != 4 + n_l:
        return

    with gp_fig.batch_update():
        gp_fig.data[0].x = x_band.tolist()
        gp_fig.data[0].y = y_lo.tolist()
        gp_fig.data[1].x = x_band.tolist()
        gp_fig.data[1].y = y_hi.tolist()
        gp_fig.data[2].x = x_band.tolist()
        gp_fig.data[2].y = y_mean.tolist()
        gp_fig.data[3].x = [0.0, 0.0]
        gp_fig.data[3].y = y_range
        err_shapes: list[dict] = []
        for i, Lval in enumerate(L_values):
            mask = L_arr == Lval
            order = np.argsort(z0[mask])
            idxs = np.flatnonzero(mask)[order]
            color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
            size, opacity, colors = _marker_style_for_L(
                Lval, order, f_share, color=color
            )
            z_L = z0[mask][order]
            phi_L = phi0[mask][order]
            sig_L = sigma_plot[mask][order]
            n_in_L = int(included[mask].sum())
            if n_in_L > 0:
                err_shapes.extend(
                    _errorbar_shapes_for_L(z_L, phi_L, sig_L, opacity, colors)
                )
            tr = gp_fig.data[4 + i]
            tr.x = z_L.tolist()
            tr.y = phi_L.tolist()
            tr.customdata = np.column_stack(
                [
                    idxs,
                    T_arr[mask][order],
                    a[mask][order],
                    f_hat[mask][order],
                    (phi_L - f_hat[mask][order]),
                    f_share[mask][order],
                    pi[mask][order],
                ]
            ).tolist()
            tr.marker.size = list(size)
            tr.marker.color = list(colors)
            tr.marker.opacity = list(opacity)
            tr.hovertemplate = _point_hovertemplate(Lval)
        gp_fig.layout.shapes = err_shapes
        gp_fig.layout.yaxis.range = y_range
        gp_fig.layout.width = _gp_plot_width()
        gp_fig.layout.title.text = (
            f"{'f₀' if _correction_on() else 'f'}(z) GP posterior ±1σ  [{state['dataset']}]  "
            f"ℓ_f={params['gp_ell']:.3g}, η_f={params['gp_eta']:.3g}"
            + (
                f", ℓ_g={params.get('disc_gp_ell', params['gp_ell']):.3g}"
                if _is_additive_form() and _coupling() != "weighted"
                else ""
            )
            + (f" · ω={OMEGA_EXACT:g}" if _correction_on() else "")
            + " · opacity=f-share"
            + f" · {_inclusion_status_text()}"
        )


def _update_ell_profile(params: dict[str, float]) -> None:
    """Refresh ΔlogL(ℓ_f) at exact exponents (independent of heatmap profiling)."""
    fig = globals().get("ell_profile_fig")
    compiled = state.get("compiled")
    if fig is None or compiled is None or _n_included() < MIN_INCLUDED_POINTS:
        return
    t0_arg, L0_arg, p_arg, q_arg = _disc_profile_slots(params)
    sigma_eff = float(params["sigma_model"]) * float(params["K"])
    ell_grid = _gp_ell_profile_grid()
    ll = np.asarray(
        profile_gp_ell(
            compiled,
            ell_grid,
            T_c=TC_EXACT,
            nu=NU_EXACT,
            beta=BETA_EXACT,
            t0=t0_arg,
            L0=L0_arg,
            p=p_arg,
            q=q_arg,
            sigma_model=sigma_eff,
            gp_eta=float(params["gp_eta"]),
            obs_sigma_scale=float(params.get("obs_sigma_scale", 1.0)),
            disc_gp_ell=float(params.get("disc_gp_ell", params["gp_ell"])),
        ),
        dtype=np.float64,
    )
    finite = np.isfinite(ll)
    if not np.any(finite):
        return
    ll_max = float(np.nanmax(ll))
    dlog = ll - ll_max
    i_star = int(np.nanargmax(ll))
    ell_star = float(ell_grid[i_star])
    ell_slider = float(params["gp_ell"])
    i_slider = int(np.argmin(np.abs(np.log(ell_grid) - np.log(max(ell_slider, 1e-12)))))
    dlog_slider = float(dlog[i_slider])
    with fig.batch_update():
        fig.data[0].x = ell_grid.tolist()
        fig.data[0].y = dlog.tolist()
        fig.layout.title.text = (
            f"ℓ_f profile at exact (Tc,ν,β)  ℓ*={ell_star:.3g}  "
            f"Δ(slider)={dlog_slider:.1f}"
        )
    # Length-1 x updates inside batch_update are dropped when they overlap
    # the line's x (FigureWidget _remove_overlapping_props).
    fig.data[1].x = [float(ell_slider)]
    fig.data[1].y = [float(dlog_slider)]
    fig.data[2].x = [float(ell_star)]
    fig.data[2].y = [float(dlog[i_star])]
    _update_kappa_profile(params)


def _build_kappa_profile_figure() -> go.FigureWidget:
    """ΔlogL vs κ at exact (Tc, ν, β). FSS form only."""
    return go.FigureWidget(
        data=[
            go.Scatter(
                x=[],
                y=[],
                mode="lines+markers",
                line=dict(color="#1f77b4", width=2),
                marker=dict(size=6),
                name="ΔlogL",
                hovertemplate="κ=%{x:.3g}<br>ΔlogL=%{y:.2f}<extra></extra>",
            ),
            go.Scatter(
                x=[],
                y=[],
                mode="markers",
                marker=dict(size=11, color="#d62728", symbol="x"),
                name="slider",
                hovertemplate="slider κ=%{x:.3g}<extra></extra>",
            ),
            go.Scatter(
                x=[],
                y=[],
                mode="markers",
                marker=dict(size=11, color="#2ca02c", symbol="diamond"),
                name="κ*",
                hovertemplate="κ*=%{x:.3g}<extra></extra>",
            ),
        ],
        layout=go.Layout(
            title=dict(text="profile κ at exact (Tc,ν,β)", font=dict(size=13)),
            xaxis=dict(title="κ (relative |t| weight in a)"),
            yaxis_title="ΔlogL",
            width=HEATMAP_WIDTH,
            height=GP_PLOT_HEIGHT,
            template="plotly_white",
            margin=dict(l=56, r=16, t=48, b=48),
            legend=dict(font=dict(size=9), yanchor="top", y=0.99, xanchor="right", x=0.99),
            hovermode="closest",
        ),
    )


def _lml_at_disc_slots(
    params: dict[str, float],
    *,
    kappa: float,
    gp_ell: float,
) -> float:
    compiled = state["compiled"]
    scales = compiled.gp_scales
    omega = float(params["omega"])
    return float(
        compiled.evaluate_with_gp_jax(
            TC_EXACT,
            NU_EXACT,
            BETA_EXACT,
            float(gp_ell),
            float(params["gp_eta"]),
            float(scales.correction_gp_ell),
            float(scales.correction_gp_eta),
            float(kappa),
            1.0,
            2.0,
            omega,
            float(params["sigma_model"]) * float(params["K"]),
            float(params.get("obs_sigma_scale", 1.0)),
            False,
            float(params.get("disc_gp_ell", gp_ell)),
        )
    )


def _update_kappa_profile(params: dict[str, float]) -> None:
    """Refresh ΔlogL(κ) at exact exponents (FSS form, when the panel is shown)."""
    fig = globals().get("kappa_profile_fig")
    compiled = state.get("compiled")
    if fig is None or compiled is None:
        return
    if not _kappa_profile_visible() or _n_included() < MIN_INCLUDED_POINTS:
        return
    k_slider = float(params["kappa"])
    k_grid = _kappa_profile_grid(k_slider)
    sigma_eff = float(params["sigma_model"]) * float(params["K"])
    if _profile_lf_on():
        ells = _gp_ell_profile_grid()
        ll = np.array(
            [
                float(
                    np.nanmax(
                        profile_gp_ell(
                            compiled,
                            ells,
                            T_c=TC_EXACT,
                            nu=NU_EXACT,
                            beta=BETA_EXACT,
                            t0=float(k),
                            L0=1.0,
                            p=2.0,
                            q=float(params["omega"]),
                            sigma_model=sigma_eff,
                            gp_eta=float(params["gp_eta"]),
                            obs_sigma_scale=float(params.get("obs_sigma_scale", 1.0)),
                            disc_gp_ell=float(params.get("disc_gp_ell", params["gp_ell"])),
                        )
                    )
                )
                for k in k_grid
            ],
            dtype=np.float64,
        )
        ell_note = f"max_ℓ ({GP_ELL_PROFILE_N}-pt)"
    else:
        ell = float(params["gp_ell"])
        ll = np.array(
            [_lml_at_disc_slots(params, kappa=float(k), gp_ell=ell) for k in k_grid],
            dtype=np.float64,
        )
        ell_note = f"ℓ_f={ell:.3g}"
    finite = np.isfinite(ll)
    if not np.any(finite):
        return
    ll_max = float(np.nanmax(ll))
    dlog = ll - ll_max
    i_star = int(np.nanargmax(ll))
    k_star = float(k_grid[i_star])
    i_slider = int(np.argmin(np.abs(k_grid - k_slider)))
    dlog_slider = float(dlog[i_slider])
    with fig.batch_update():
        fig.data[0].x = k_grid.tolist()
        fig.data[0].y = dlog.tolist()
        fig.layout.title.text = (
            f"κ profile at exact (Tc,ν,β)  κ*={k_star:.3g}  "
            f"Δ(slider)={dlog_slider:.1f}  · {ell_note}"
        )
    # Length-1 x updates inside batch_update are dropped when they overlap
    # the line's x (FigureWidget _remove_overlapping_props).
    fig.data[1].x = [float(k_slider)]
    fig.data[1].y = [float(dlog_slider)]
    fig.data[2].x = [float(k_star)]
    fig.data[2].y = [float(dlog[i_star])]


def _update_collapse(params: dict[str, float]) -> None:
    a, sigma_plot = _sigma_eff_phi(params)
    f_hat = _f_hat_phi_m(params)
    f_share, pi = _collapse_share_and_pi(params, f_hat)
    z0 = state["z0"]
    phi0 = state["phi0"]
    L_arr = state["L_arr"]
    T_arr = state["T_arr"]
    L_values = state["L_values"]
    Y_LO = state["Y_LO"]
    Y_HI = state["Y_HI"]
    included = np.asarray(state["included"], dtype=bool)
    global _updating_from_code
    n_clip = int(np.sum((phi0 - sigma_plot < Y_LO) | (phi0 + sigma_plot > Y_HI)))
    bar_label = "σ_MC" if _is_additive_form() else "σ_eff"
    _updating_from_code = True
    try:
        err_shapes: list[dict] = []
        with collapse_fig.batch_update():
            for i, Lval in enumerate(L_values):
                mask = L_arr == Lval
                order = np.argsort(z0[mask])
                idxs = np.flatnonzero(mask)[order]
                color = PLOTLY_COLORS[i % len(PLOTLY_COLORS)]
                size, opacity, colors = _marker_style_for_L(
                    Lval, order, f_share, color=color
                )
                tr = collapse_fig.data[i]
                # FigureWidget requires plain Python lists (numpy arrays break
                # plotly.basewidget._remove_overlapping_props via `if not arr`).
                z_L = z0[mask][order]
                phi_L = phi0[mask][order]
                sig_L = sigma_plot[mask][order]
                tr.x = z_L.tolist()
                tr.y = phi_L.tolist()
                tr.customdata = np.column_stack(
                    [
                        idxs,
                        T_arr[mask][order],
                        a[mask][order],
                        f_hat[mask][order],
                        (phi_L - f_hat[mask][order]),
                        f_share[mask][order],
                        pi[mask][order],
                    ]
                ).tolist()
                tr.error_y.visible = False
                tr.marker.size = list(size)
                tr.marker.opacity = list(opacity)
                tr.marker.color = list(colors)
                # Keep legend visibility in sync with whether any L points are included.
                n_in_L = int(included[mask].sum())
                want_visible: bool | str = True if n_in_L > 0 else "legendonly"
                if tr.visible != want_visible:
                    tr.visible = want_visible
                if n_in_L > 0:
                    err_shapes.extend(
                        _errorbar_shapes_for_L(z_L, phi_L, sig_L, opacity, colors)
                    )
            guide = collapse_fig.data[len(L_values)]
            guide.y = [Y_LO, Y_HI]
            collapse_fig.layout.yaxis.range = [Y_LO, Y_HI]
            collapse_fig.layout.shapes = err_shapes
            collapse_fig.layout.title.text = (
                f"m collapse (LML: {_channel_tag()}) [{state['dataset']}] + {bar_label} "
                f"(K={params['K']:.3g}, t0={params['t0']:.3g}, L0={params['L0']:.3g}, "
                f"p={params['p']:.3g}, q={params['q']:.3g}, "
                f"ω={params['omega']:.3g}, κ={params['kappa']:.3g}, "
                f"σ={params['sigma_model']:.3g}; "
                f"{n_clip}/{len(phi0)} bars clipped; {_inclusion_status_text()})"
            )
    finally:
        _updating_from_code = False


def _beta_is_free() -> bool:
    w = globals().get("checkbox_beta_free")
    return bool(w is not None and w.value)


def _profile_lf_on() -> bool:
    w = globals().get("checkbox_profile_lf")
    return bool(w is not None and w.value)


def _kappa_profile_visible() -> bool:
    return state.get("disc_form") == "additive_gp_fss"


def _kappa_profile_grid(kappa_slider: float | None = None) -> np.ndarray:
    lo = float(DISCREPANCY_KAPPA_PRIOR_LOWER)
    hi = float(KAPPA_SLIDER_MAX)
    grid = np.linspace(lo, hi, KAPPA_PROFILE_N, dtype=np.float64)
    if kappa_slider is None:
        return grid
    k = float(kappa_slider)
    if lo <= k <= hi and not np.any(np.isclose(grid, k, rtol=0.0, atol=1e-10)):
        grid = np.sort(np.append(grid, k))
    return grid


def _gp_ell_profile_grid() -> np.ndarray:
    return default_gp_ell_profile_grid(
        GP_ELL_SLIDER_MIN, GP_ELL_SLIDER_MAX, GP_ELL_PROFILE_N
    )


def _heatmap_gp_ell_kwargs(params: dict[str, float]) -> dict:
    """Scalar ℓ_f, or a log-grid when **profile ℓ_f** is checked."""
    kwargs: dict = {"gp_eta": float(params["gp_eta"])}
    if _profile_lf_on():
        kwargs["gp_ell_grid"] = _gp_ell_profile_grid()
    else:
        kwargs["gp_ell"] = float(params["gp_ell"])
    return kwargs


def _heatmap_zmin_view(
    z_plot: np.ndarray,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    x_range: list[float],
    y_range: list[float],
) -> float:
    in_x = (x_grid >= x_range[0]) & (x_grid <= x_range[1])
    in_y = (y_grid >= y_range[0]) & (y_grid <= y_range[1])
    z_view = z_plot[np.ix_(in_y, in_x)]
    above_floor = z_view[z_view > HEATMAP_Z_FLOOR + 1e-9]
    if above_floor.size >= 2:
        return float(max(HEATMAP_Z_FLOOR, np.nanmin(above_floor)))
    if above_floor.size == 1:
        return float(max(HEATMAP_Z_FLOOR, above_floor[0] - 5.0))
    return HEATMAP_Z_FLOOR


def _compute_joint_heatmap(
    *,
    pair: str,
    x_grid: np.ndarray,
    y_grid: np.ndarray,
    T_c: float,
    nu: float,
    beta: float,
    x_exact: float,
    y_exact: float,
    params: dict[str, float],
) -> dict:
    t0_arg, L0_arg, p_arg, q_arg = _disc_profile_slots(params)
    sigma_eff = float(params["sigma_model"]) * float(params["K"])
    gp_eta = float(params["gp_eta"])
    disc_kw = dict(
        pair=pair,
        T_c=T_c,
        nu=nu,
        beta=beta,
        t0=t0_arg,
        L0=L0_arg,
        p=p_arg,
        q=q_arg,
        sigma_model=sigma_eff,
        gp_eta=gp_eta,
        obs_sigma_scale=float(params.get("obs_sigma_scale", 1.0)),
        disc_gp_ell=float(params.get("disc_gp_ell", params["gp_ell"])),
    )
    # Loop over ℓ_f in Python so we reuse the scalar heatmap JIT (a nested
    # vmap over a 17-point grid recompiles and can stall the kernel).
    if _profile_lf_on():
        log_ml = None
        for ell in _gp_ell_profile_grid():
            z_ell = profile_joint_with_disc(
                state["compiled"],
                x_grid,
                y_grid,
                gp_ell=float(ell),
                **disc_kw,
            )
            log_ml = z_ell if log_ml is None else np.maximum(log_ml, z_ell)
    else:
        log_ml = profile_joint_with_disc(
            state["compiled"],
            x_grid,
            y_grid,
            gp_ell=float(params["gp_ell"]),
            **disc_kw,
        )
    z_raw = np.asarray(log_ml, dtype=np.float64)
    z_max = float(np.nanmax(z_raw))
    z = z_raw - z_max
    z_plot = np.maximum(z, HEATMAP_Z_FLOOR)
    j_max, i_max = np.unravel_index(int(np.nanargmax(z)), z.shape)
    x_mle = float(x_grid[i_max])
    y_mle = float(y_grid[j_max])
    i_exact = int(np.argmin(np.abs(x_grid - x_exact)))
    j_exact = int(np.argmin(np.abs(y_grid - y_exact)))
    dlog_exact = float(z[j_exact, i_exact])
    x_range, y_range, zoom_level, n_zoom, exact_in_view = _heatmap_axis_ranges(
        z_plot, x_grid, y_grid, x_mle, y_mle, x_exact, y_exact, dlog_exact
    )
    zmin_view = _heatmap_zmin_view(z_plot, x_grid, y_grid, x_range, y_range)
    return {
        "z_plot": z_plot,
        "zmin_view": zmin_view,
        "x_mle": x_mle,
        "y_mle": y_mle,
        "dlog_exact": dlog_exact,
        "z_max": z_max,
        "x_range": x_range,
        "y_range": y_range,
        "zoom_level": zoom_level,
        "n_zoom": n_zoom,
        "exact_in_view": exact_in_view,
        "x_grid": x_grid,
        "y_grid": y_grid,
    }


def _paint_joint_heatmap(
    fig: go.FigureWidget,
    result: dict,
    *,
    x_exact: float,
    y_exact: float,
    title: str,
) -> None:
    x_grid = np.asarray(result["x_grid"], dtype=np.float64)
    y_grid = np.asarray(result["y_grid"], dtype=np.float64)
    z_plot = result["z_plot"]
    with fig.batch_update():
        fig.data[0].x = x_grid.tolist()
        fig.data[0].y = y_grid.tolist()
        fig.data[0].z = z_plot.tolist()
        fig.data[0].zmin = result["zmin_view"]
        fig.data[0].zmax = 0.0
        fig.data[1].x = x_grid.tolist()
        fig.data[1].y = y_grid.tolist()
        fig.data[1].z = z_plot.tolist()
        fig.data[2].x = [x_exact]
        fig.data[2].y = [y_exact]
        fig.data[3].x = [result["x_mle"]]
        fig.data[3].y = [result["y_mle"]]
        fig.layout.xaxis.range = result["x_range"]
        fig.layout.yaxis.range = result["y_range"]
        fig.layout.title.text = title


def _update_heatmap(params: dict[str, float]) -> None:
    if state["compiled"] is None or _n_included() < MIN_INCLUDED_POINTS:
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}). Click legend / points to include more.</span>"
        )
        return
    t_start = time.perf_counter()
    if _profile_lf_on():
        status.value = (
            f"profiling ℓ_f ({GP_ELL_PROFILE_N}-pt log grid) · {_inclusion_status_html()}…"
        )
    k_scale = float(params["K"])
    sigma_slider = float(params["sigma_model"])
    sigma_eff = sigma_slider * k_scale
    form_tag = _form_tag()
    sigma_name = (
        "σ_g"
        if _is_additive_form() and _coupling() != "weighted"
        else "σ_model"
    )
    if _is_additive_form() and _coupling() == "weighted":
        sigma_name = "σ_g (unused)"
    k_warn = ""
    if k_scale <= 0.0:
        if _is_additive_form() and _is_gated_coupling() and _coupling() == "mixture":
            k_warn = (
                " <span style='color:#c44e52'><b>K=0 zeros g</b> "
                "(f still gated by π)</span>"
            )
        else:
            k_warn = (
                " <span style='color:#c44e52'><b>K=0 ⇒ discrepancy off</b> "
                "(σ slider ignored)</span>"
            )
    corr_note = (
        f" · f₁ on (ω={OMEGA_EXACT:g})"
        if _correction_on()
        else ""
    )
    add_off = (
        " <span style='color:#c44e52'>additive a·g mean off while f₁ is on"
        " (noise inflation still stacks)</span>"
        if _correction_on() and _is_additive_form()
        else ""
    )

    tc_res = _compute_joint_heatmap(
        pair="T_c_nu",
        x_grid=tc_grid,
        y_grid=nu_grid,
        T_c=0.0,
        nu=0.0,
        beta=BETA_EXACT,
        x_exact=TC_EXACT,
        y_exact=NU_EXACT,
        params=params,
    )
    ell_tag = (
        f"ℓ_f profiled (n={GP_ELL_PROFILE_N})"
        if _profile_lf_on()
        else f"ℓ={params['gp_ell']:.3g}"
    )
    _paint_joint_heatmap(
        heatmap_fig,
        tc_res,
        x_exact=TC_EXACT,
        y_exact=NU_EXACT,
        title=(
            f"[{state['dataset']}|{form_tag}|n={_n_included()}] "
            f"{ell_tag} η={params['gp_eta']:.3g} · "
            f"{sigma_name}={sigma_slider:.3g} K={k_scale:.3g} → η_disc={sigma_eff:.3g} · "
            f"MLE Tc={tc_res['x_mle']:.5f}, ν={tc_res['y_mle']:.4g} · "
            f"exact Δ={tc_res['dlog_exact']:.1f} maxL={tc_res['z_max']:.1f}"
        ),
    )

    beta_note = ""
    if _beta_is_free():
        beta_res = _compute_joint_heatmap(
            pair="nu_beta",
            x_grid=nu_grid,
            y_grid=beta_grid,
            T_c=TC_EXACT,
            nu=0.0,
            beta=0.0,
            x_exact=NU_EXACT,
            y_exact=BETA_EXACT,
            params=params,
        )
        _paint_joint_heatmap(
            heatmap_beta_fig,
            beta_res,
            x_exact=NU_EXACT,
            y_exact=BETA_EXACT,
            title=(
                f"[{state['dataset']}|{form_tag}|n={_n_included()}] "
                f"Tc={TC_EXACT:.5g} · "
                f"MLE ν={beta_res['x_mle']:.4g}, β={beta_res['y_mle']:.4g} · "
                f"exact Δ={beta_res['dlog_exact']:.1f} maxL={beta_res['z_max']:.1f}"
            ),
        )
        beta_note = (
            f" · (ν,β) MLE ν={beta_res['x_mle']:.5f}, β={beta_res['y_mle']:.5f} "
            f"ΔlogL(exact)={beta_res['dlog_exact']:.2f}"
        )

    elapsed = time.perf_counter() - t_start
    sharp = (
        "sharp peak — zoomed"
        if tc_res["n_zoom"] < HEATMAP_MIN_ZOOM_CELLS
        else f"zoom ΔlogL>{tc_res['zoom_level']:g}"
    )
    exact_note = (
        "exact in view" if tc_res["exact_in_view"] else "exact off-zoom (far below peak)"
    )
    ell_status = (
        "ℓ_f <b>profiled</b>" if _profile_lf_on() else f"ℓ_f={params['gp_ell']:.3g}"
    )
    status.value = (
        f"<b>{state['dataset']}</b> · <b>{form_tag}</b> · form=<code>{state['disc_form']}</code> · "
        f"{_inclusion_status_html()} · "
        f"{sigma_name}={sigma_slider:.3g} · K={k_scale:.3g} · η_disc={sigma_eff:.3g} · "
        f"s_σ={float(params.get('obs_sigma_scale', 1.0)):.3g} · "
        f"{ell_status} · "
        f"ℓ_g={float(params.get('disc_gp_ell', params['gp_ell'])):.3g} · "
        f"η_f={params['gp_eta']:.3g} · "
        f"heatmap {elapsed:.2f}s · "
        f"MLE Tc={tc_res['x_mle']:.6f}, ν={tc_res['y_mle']:.5f} · "
        f"ΔlogL(exact)={tc_res['dlog_exact']:.2f} · maxL={tc_res['z_max']:.1f} ({exact_note}) · "
        f"{sharp} ({tc_res['n_zoom']} cells){beta_note}{corr_note}{add_off}{k_warn}"
    )


def _apply_inclusion_change(*, refresh_collapse: bool = True) -> None:
    """Recompile + refresh heatmap (and optionally collapse styling) after mask edits."""
    params = _disc_params()
    if not _recompile_active(quiet=True):
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}).</span>"
        )
        if refresh_collapse:
            _update_collapse(params)
            _update_gp(params)
            _update_ell_profile(params)
        return
    status.value = f"updating heatmap ({_inclusion_status_html()})…"
    if refresh_collapse:
        _update_collapse(params)
        _update_gp(params)
        _update_ell_profile(params)
    _update_heatmap(params)


# --- interactive selection on collapse plot ---
_updating_from_code = False
_ui_ready = False
_refreshing = False


def _trace_is_visible(visible) -> bool:
    return visible is True or visible is None


def _on_legend_visibility_change(trace, points_unused=None) -> None:
    """Legend click hides/shows an L-trace → include/exclude all points of that L."""
    global _updating_from_code
    if _updating_from_code:
        return
    # Map trace → L via name "L=..."
    name = str(getattr(trace, "name", "") or "")
    if not name.startswith("L="):
        return
    try:
        Lval = int(name.split("=", 1)[1])
    except ValueError:
        return
    include = _trace_is_visible(trace.visible)
    L_arr = state["L_arr"]
    mask = L_arr == Lval
    included = np.asarray(state["included"], dtype=bool)
    if bool(np.all(included[mask] == include)):
        return
    included[mask] = include
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _on_point_click(trace, points, selector) -> None:
    """Click a marker to toggle that point's inclusion (grey = excluded)."""
    if _updating_from_code or not points.point_inds:
        return
    # customdata columns: [idx, T, a]
    cd = np.asarray(trace.customdata)
    local = int(points.point_inds[0])
    idx = int(cd[local, 0])
    included = np.asarray(state["included"], dtype=bool).copy()
    included[idx] = not included[idx]
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _on_box_selection(trace, points, selector) -> None:
    """Box/lasso select: toggle inclusion for all selected points on this trace."""
    if _updating_from_code or not points.point_inds:
        return
    cd = np.asarray(trace.customdata)
    idxs = [int(cd[i, 0]) for i in points.point_inds]
    included = np.asarray(state["included"], dtype=bool).copy()
    # If any selected point is included, exclude all; else include all (toggle group).
    if any(included[i] for i in idxs):
        for i in idxs:
            included[i] = False
    else:
        for i in idxs:
            included[i] = True
    state["included"] = included
    _apply_inclusion_change(refresh_collapse=True)


def _bind_collapse_interactions(fig: go.FigureWidget) -> None:
    L_values = state["L_values"]
    for i in range(len(L_values)):
        tr = fig.data[i]
        tr.on_click(_on_point_click)
        tr.on_selection(_on_box_selection)
        tr.on_change(_on_legend_visibility_change, "visible")


def _include_all_points(_btn=None) -> None:
    state["included"] = np.ones(len(state["df_full"]), dtype=bool)
    _apply_inclusion_change(refresh_collapse=True)


def _sync_figures_box() -> None:
    if _beta_is_free():
        figures_box.children = (collapse_fig, heatmap_fig, heatmap_beta_fig)
    else:
        figures_box.children = (collapse_fig, heatmap_fig)


def _sync_gp_box() -> None:
    gp = globals().get("gp_fig")
    ell = globals().get("ell_profile_fig")
    kap = globals().get("kappa_profile_fig")
    box = globals().get("gp_box")
    if box is None or gp is None or ell is None:
        return
    if _kappa_profile_visible() and kap is not None:
        box.children = (gp, widgets.HBox([ell, kap]))
    else:
        box.children = (gp, ell)


def _rebuild_figures_and_refresh() -> None:
    global collapse_fig, heatmap_fig, heatmap_beta_fig, gp_fig, ell_profile_fig, kappa_profile_fig, _updating_from_code
    _updating_from_code = True
    try:
        collapse_fig = _build_collapse_figure()
        heatmap_fig = _build_heatmap_figure()
        heatmap_beta_fig = _build_beta_heatmap_figure()
        gp_fig = _build_gp_figure()
        ell_profile_fig = _build_ell_profile_figure()
        kappa_profile_fig = _build_kappa_profile_figure()
        _sync_figures_box()
        _sync_gp_box()
    finally:
        _updating_from_code = False
    params = _disc_params()
    _update_collapse(params)
    _update_gp(params)
    _update_ell_profile(params)
    _update_heatmap(params)


def _sync_channel_controls() -> None:
    avail = state.get("available_channels") or {}
    ch = state.get("channels") or {}
    boxes = (
        ("m", globals().get("checkbox_ch_m")),
        ("m2", globals().get("checkbox_ch_m2")),
        ("m4", globals().get("checkbox_ch_m4")),
        ("binder", globals().get("checkbox_ch_binder")),
    )
    if any(box is None for _, box in boxes):
        return
    global _updating_from_code
    prev = _updating_from_code
    _updating_from_code = True
    try:
        for key, box in boxes:
            has = bool(avail.get(key, False))
            box.disabled = not has
            box.value = bool(ch.get(key, False)) and has
    finally:
        _updating_from_code = prev


def _sync_model_controls() -> None:
    if _is_additive_form():
        slider_sigma.description = "σ_g"
    else:
        slider_sigma.description = "σ_model"
    form = state["disc_form"]
    gp_form = _is_additive_form(form)
    use_amp = form not in ("additive_gp_z_threshold", "additive_gp_fss")
    use_fss = form == "additive_gp_fss"
    coup_w = globals().get("dropdown_coupling")
    if coup_w is not None:
        coup_w.disabled = not gp_form
    for s in (slider_t0, slider_L0, slider_p, slider_q):
        s.disabled = not use_amp
    slider_omega.disabled = not use_fss
    slider_kappa.disabled = not use_fss
    slider_sigma.disabled = (not gp_form and form != "noise") or (
        gp_form and _coupling() == "weighted"
    )
    disc_ell_w = globals().get("slider_disc_ell")
    if disc_ell_w is not None:
        disc_ell_w.disabled = (not gp_form) or (_coupling() == "weighted")
    _sync_channel_controls()
    _sync_gp_box()


def _on_slider_change(change=None) -> None:
    global _refreshing
    if (not _ui_ready) or _refreshing or _updating_from_code:
        return
    _refreshing = True
    try:
        params = _disc_params()
        status.value = "updating…"
        _update_collapse(params)
        _update_gp(params)
        _update_ell_profile(params)
        # Heatmaps already max over ℓ_f: moving the slider only affects the GP panel.
        owner = getattr(change, "owner", None)
        if owner is None and isinstance(change, dict):
            owner = change.get("owner")
        skip_heatmap = _profile_lf_on() and owner is slider_gp_ell
        if skip_heatmap:
            status.value = (
                f"{_inclusion_status_html()} · ℓ_f slider={params['gp_ell']:.3g} "
                "(heatmap profiled; slider is GP panel only)"
            )
            return
        _update_heatmap(params)
    finally:
        _refreshing = False


def _on_beta_free_change(_change=None) -> None:
    if not _ui_ready:
        return
    _sync_figures_box()
    status.value = "updating heatmap…"
    _update_heatmap(_disc_params())


def _on_correction_change(_change=None) -> None:
    if not _ui_ready:
        return
    on = bool(checkbox_correction.value)
    if bool(state.get("correction", False)) == on and _correction_on() == on:
        return
    status.value = (
        f"recompiling with L<sup>-ω</sup> f₁ (ω={OMEGA_EXACT:g})…"
        if on
        else "recompiling without correction…"
    )
    state["config"] = make_config(
        str(state["disc_form"]),
        correction=on,
        channels=state.get("channels"),
    )
    state["correction"] = on
    if not _recompile_active(quiet=True):
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}).</span>"
        )
        return
    params = _disc_params()
    _update_collapse(params)
    _update_gp(params)
    _update_ell_profile(params)
    _update_heatmap(params)


def _on_channel_change(_change=None) -> None:
    if (not _ui_ready) or _updating_from_code:
        return
    avail = state.get("available_channels") or {}
    ch = _resolve_channels(avail)
    if ch == state.get("channels"):
        _sync_channel_controls()
        return
    status.value = f"recompiling LML ({_channel_tag(ch)})…"
    state["channels"] = ch
    state["config"] = make_config(
        str(state["disc_form"]),
        correction=_correction_requested(),
        channels=ch,
    )
    _sync_channel_controls()
    if not _recompile_active(quiet=True):
        status.value = (
            f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
            f"({_inclusion_status_html()}).</span>"
        )
        return
    params = _disc_params()
    _update_collapse(params)
    _update_gp(params)
    _update_ell_profile(params)
    _update_heatmap(params)


def _on_dataset_or_model_change(_change=None) -> None:
    if not _ui_ready:
        return
    family, name = dropdown_dataset.value
    form = str(dropdown_model.value)
    coup = str(dropdown_coupling.value)
    same_data = (
        name == state.get("dataset")
        and family == state.get("family")
        and form == state.get("disc_form")
    )
    if same_data and coup == state.get("disc_coupling"):
        return
    if same_data:
        status.value = f"recompiling coupling <b>{coup}</b>…"
        state["disc_coupling"] = coup
        state["config"] = make_config(
            form,
            correction=_correction_requested(),
            channels=state.get("channels"),
            coupling=coup,
        )
        _sync_model_controls()
        if not _recompile_active(quiet=True):
            status.value = (
                f"<span style='color:#c44e52'>Need ≥{MIN_INCLUDED_POINTS} included points "
                f"({_inclusion_status_html()}).</span>"
            )
            return
        params = _disc_params()
        _update_collapse(params)
        _update_gp(params)
        _update_ell_profile(params)
        _update_heatmap(params)
        return
    status.value = f"loading <b>{family}/{name}</b> / <b>{form}</b> / <b>{coup}</b>…"
    try:
        load_dataset(
            name,
            family=family,
            discrepancy_form=form,
            coupling=coup,
            quiet=True,
        )
        _sync_model_controls()
        dataset_info.value = (
            f"<i>{_get_dataset(family, name).description}</i> "
            f"({family}/{name}: {len(state['df_full'])} points, "
            f"L={list(state['L_values'])})"
        )
        # Heatmap uses rebuilt tc_grid after family switch.
        _rebuild_figures_and_refresh()
    except Exception as exc:
        status.value = (
            f"<span style='color:#c44e52'>Failed to load {family}/{name}/{form}/{coup}: {exc}</span>"
        )
        raise


slider_kwargs = dict(continuous_update=False, readout_format=".3g")
dropdown_dataset = widgets.Dropdown(
    options=AVAILABLE_DATASETS,
    value=(state["family"], state["dataset"]),
    description="dataset",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "70px"},
)
dropdown_model = widgets.Dropdown(
    options=DISC_FORM_OPTIONS,
    value=state["disc_form"],
    description="model",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "70px"},
)
dropdown_coupling = widgets.Dropdown(
    options=COUPLING_OPTIONS,
    value=state.get("disc_coupling", DEFAULT_COUPLING),
    description="coupling",
    layout=widgets.Layout(width="240px"),
    style={"description_width": "70px"},
    tooltip=(
        "How a enters the GP (ignored for noise inflation). "
        "additive: Φ=f+a g. mixture: Φ=(1-π)f+π g with π=a/(1+a). "
        "weighted: Φ=(1-π)f (no g)."
    ),
)
dataset_info = widgets.HTML(
    value=(
        f"<i>{_get_dataset(state['family'], state['dataset']).description}</i> "
        f"({state['family']}/{state['dataset']}: {len(state['df_full'])} points, "
        f"L={list(state['L_values'])})"
    )
)

slider_t0 = widgets.FloatSlider(
    value=DISCREPANCY_T0_INIT,
    min=DISCREPANCY_T0_PRIOR_LOWER,
    max=DISCREPANCY_T0_PRIOR_UPPER,
    step=0.01,
    description="t0",
    **slider_kwargs,
)
slider_L0 = widgets.FloatSlider(
    value=DISCREPANCY_L0_INIT,
    min=DISCREPANCY_L0_PRIOR_LOWER,
    max=DISCREPANCY_L0_PRIOR_UPPER,
    step=1.0,
    description="L0",
    **slider_kwargs,
)
slider_p = widgets.FloatSlider(
    value=DISCREPANCY_P_INIT,
    min=DISCREPANCY_P_PRIOR_LOWER,
    max=DISCREPANCY_P_PRIOR_UPPER,
    step=0.05,
    description="p",
    **slider_kwargs,
)
slider_q = widgets.FloatSlider(
    value=DISCREPANCY_Q_INIT,
    min=DISCREPANCY_Q_PRIOR_LOWER,
    max=DISCREPANCY_Q_PRIOR_UPPER,
    step=0.05,
    description="q",
    **slider_kwargs,
)
slider_omega = widgets.FloatSlider(
    value=float(OMEGA_EXACT),
    min=float(OMEGA_PRIOR_LOWER),
    max=float(OMEGA_PRIOR_UPPER),
    step=0.05,
    description="ω",
    **slider_kwargs,
)
slider_kappa = widgets.FloatSlider(
    value=DISCREPANCY_KAPPA_INIT,
    min=DISCREPANCY_KAPPA_PRIOR_LOWER,
    max=KAPPA_SLIDER_MAX,
    step=0.1,
    description="κ",
    **slider_kwargs,
)
slider_sigma = widgets.FloatSlider(
    value=SIGMA_G_INIT,
    min=0.0,
    max=SIGMA_SLIDER_MAX,
    step=0.05,
    description="σ_model",
    **slider_kwargs,
)
slider_K = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=1.0,
    step=0.01,
    description="K",
    **slider_kwargs,
)
slider_gp_ell = widgets.FloatSlider(
    value=GP_ELL_INIT,
    min=GP_ELL_SLIDER_MIN,
    max=GP_ELL_SLIDER_MAX,
    step=0.01,
    description="ℓ_f",
    **slider_kwargs,
)
slider_disc_ell = widgets.FloatSlider(
    value=GP_ELL_INIT,
    min=GP_ELL_SLIDER_MIN,
    max=GP_ELL_SLIDER_MAX,
    step=0.01,
    description="ℓ_g",
    tooltip="Length scale of discrepancy GP g in z-units (independent of ℓ_f).",
    **slider_kwargs,
)
slider_gp_eta = widgets.FloatSlider(
    value=GP_ETA_INIT,
    min=GP_ETA_SLIDER_MIN,
    max=GP_ETA_SLIDER_MAX,
    step=0.05,
    description="η_f",
    **slider_kwargs,
)
slider_obs_sigma = widgets.FloatSlider(
    value=OBS_SIGMA_SCALE_INIT,
    min=OBS_SIGMA_SCALE_SLIDER_MIN,
    max=OBS_SIGMA_SCALE_SLIDER_MAX,
    step=0.25,
    description="s_σ",
    tooltip=(
        "Uniform multiplier on MC observation std. "
        "1 = reported σ_MC; >1 tests underestimated noise. "
        "Applies to every point (unlike discrepancy a)."
    ),
    **slider_kwargs,
)
_sync_model_controls()

btn_include_all = widgets.Button(
    description="Include all points",
    tooltip="Reset inclusion mask to all points",
    layout=widgets.Layout(width="160px"),
)
btn_include_all.on_click(_include_all_points)

checkbox_beta_free = widgets.Checkbox(
    value=False,
    description="β free",
    indent=False,
    tooltip="When on, also show (ν, β) heatmap at exact Tc",
    layout=widgets.Layout(width="110px"),
)
checkbox_beta_free.observe(_on_beta_free_change, names="value")

checkbox_profile_lf = widgets.Checkbox(
    value=False,
    description="profile ℓ_f",
    indent=False,
    tooltip=(
        "Heatmap cells use max over ℓ_f on a log grid "
        f"[{GP_ELL_SLIDER_MIN:g}, {GP_ELL_SLIDER_MAX:g}] ({GP_ELL_PROFILE_N} pts). "
        "The ℓ_f slider then only affects the GP panel."
    ),
    layout=widgets.Layout(width="130px"),
)
checkbox_profile_lf.observe(_on_slider_change, names="value")

checkbox_correction = widgets.Checkbox(
    value=bool(state.get("correction", False)),
    description="L^{-ω} f₁",
    indent=False,
    tooltip=(
        f"Leading Wegner correction Φ=f₀+L^{{-ω}}f₁ with ω=ω_exact={OMEGA_EXACT:g}. "
        "Recompiles the JAX likelihood."
    ),
    layout=widgets.Layout(width="130px"),
)
checkbox_correction.observe(_on_correction_change, names="value")

_ch_kw = dict(indent=False, layout=widgets.Layout(width="70px"))
checkbox_ch_m = widgets.Checkbox(
    value=bool((state.get("channels") or {}).get("m", True)),
    description="m",
    tooltip="Include magnetization in the heatmap LML (recompiles).",
    **_ch_kw,
)
checkbox_ch_m2 = widgets.Checkbox(
    value=bool((state.get("channels") or {}).get("m2", True)),
    description="m²",
    tooltip="Include ⟨m²⟩ in the heatmap LML (recompiles).",
    **_ch_kw,
)
checkbox_ch_m4 = widgets.Checkbox(
    value=bool((state.get("channels") or {}).get("m4", False)),
    description="m⁴",
    tooltip="Include ⟨m⁴⟩ in the heatmap LML (recompiles).",
    **_ch_kw,
)
checkbox_ch_binder = widgets.Checkbox(
    value=bool((state.get("channels") or {}).get("binder", False)),
    description="U₄",
    tooltip="Include Binder cumulant in the heatmap LML (recompiles). Binder-only: β does not enter.",
    **_ch_kw,
)
for _box in (checkbox_ch_m, checkbox_ch_m2, checkbox_ch_m4, checkbox_ch_binder):
    _box.observe(_on_channel_change, names="value")
_sync_channel_controls()

status = widgets.HTML(value="")
help_html = widgets.HTML(
    value=(
        "<b>Selection</b>: legend click toggles all points of that <code>L</code> "
        "(updates heatmap). Click a marker to include/exclude one point (grey = out). "
        "Box/lasso-drag to toggle a group. "
        "<b>Discrepancy</b>: noise / additive a·g use a=(|t|/t0)^p+(L0/L)^q. "
        "Z-threshold model: a=1 if |z|&gt;10 else 0 (t0,L0,p,q unused). "
        "FSS model: collapsed a=L<sup>-ω</sup>+κ|t|<sup>ων</sup> "
        "(ω, κ sliders; t0,L0,p,q unused). "
        "Noise: σ_eff²=(s_σ σ_MC)²+(σ_model K a)². "
        "Additive: Φ=f+(… )g with amplitude σ_g (same slider). "
        "Try large <b>σ_g</b> to absorb gated points. "
        "<b>s_σ</b> multiplies every MC std (1=reported σ_MC). "
        "<b>ℓ_f</b> / <b>η_f</b> are the universal GP on f (length scale, amplitude). "
        "<b>ℓ_g</b> is the length scale of g (z-units; unused for noise / weighted). "
        "Check <b>profile ℓ_f</b> so each heatmap cell is max over ℓ_f "
        "(exponents no longer depend on the hand-chosen slider). "
        "The ΔlogL(ℓ_f) panel is at exact (Tc,ν,β). "
        "FSS form also shows ΔlogL(κ) at exact (Tc,ν,β) (green diamond = κ*). "
        "<b>K</b> scales the discrepancy (0=off, 1=full; additive/mixture: σ_g). "
        "<b>coupling</b> (GP forms): additive Φ=f+ag; mixture (1-π)f+πg; "
        "weighted (1-π)f with π=a/(1+a). "
        "Check <b>L<sup>-ω</sup> f₁</b> for Φ=f₀+L<sup>-ω</sup>f₁ with "
        f"ω=ω_exact={OMEGA_EXACT:g} (recompiles; not the discrepancy ω slider). "
        "Check <b>β free</b> to also profile (ν, β) at exact Tc. "
        "LML <b>channels</b> (m, m², m⁴, U₄) recompile the heatmap likelihood. "
        "Collapse opacity (markers and error bars) is |f|/(|f|+|Φ−f|) (f-share of Φ)."
    )
)

collapse_fig = _build_collapse_figure()
heatmap_fig = _build_heatmap_figure()
heatmap_beta_fig = _build_beta_heatmap_figure()
gp_fig = _build_gp_figure()
ell_profile_fig = _build_ell_profile_figure()
kappa_profile_fig = _build_kappa_profile_figure()
figures_box = widgets.HBox([collapse_fig, heatmap_fig])
gp_box = widgets.VBox([gp_fig, ell_profile_fig])
_sync_gp_box()

for s in (
    slider_t0,
    slider_L0,
    slider_p,
    slider_q,
    slider_omega,
    slider_kappa,
    slider_sigma,
    slider_K,
    slider_gp_ell,
    slider_disc_ell,
    slider_gp_eta,
    slider_obs_sigma,
):
    s.observe(_on_slider_change, names="value")
dropdown_dataset.observe(_on_dataset_or_model_change, names="value")
dropdown_model.observe(_on_dataset_or_model_change, names="value")
dropdown_coupling.observe(_on_dataset_or_model_change, names="value")

controls = widgets.VBox(
    [
        widgets.HBox(
            [
                dropdown_dataset,
                dropdown_model,
                dropdown_coupling,
                checkbox_correction,
                checkbox_profile_lf,
                checkbox_beta_free,
                btn_include_all,
            ]
        ),
        dataset_info,
        widgets.HBox(
            [
                widgets.HTML("<b>LML&nbsp;</b>"),
                checkbox_ch_m,
                checkbox_ch_m2,
                checkbox_ch_m4,
                checkbox_ch_binder,
            ]
        ),
        help_html,
        widgets.HBox([slider_K, slider_sigma, slider_t0, slider_omega]),
        widgets.HBox([slider_L0, slider_p, slider_q, slider_kappa]),
        widgets.HBox([slider_gp_ell, slider_disc_ell, slider_gp_eta, slider_obs_sigma]),
        status,
    ]
)
ui = widgets.VBox([controls, figures_box, gp_box])
display(ui)

_update_collapse(_disc_params())
_update_gp(_disc_params())
_update_ell_profile(_disc_params())
_update_heatmap(_disc_params())
_ui_ready = True
